# SENTINEL on Colab
Runtime → Change runtime type → **T4 GPU** (free tier works via 4-bit; L4/A100 can drop `SENTINEL_HF_QUANT`).

In [ ]:
!nvidia-smi -L

## 1 · Install

In [ ]:
!pip -q install uv
!git clone -q https://github.com/rayen-mansouri/Indaba.git
%cd Indaba
!uv sync --python 3.12 --extra hf
!uv pip install -q bitsandbytes accelerate

## 2 · Download Qwen3-8B weights (~16 GB, once per session)
The adapter is offline-only (`local_files_only=True`), so weights must be in the HF cache first.

In [ ]:
!.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('Qwen/Qwen3-8B'))"

## 3 · Sanity check (mock model, no GPU needed)

In [ ]:
!.venv/bin/sentinel run --scenario scenarios/public/finance/finance_false_approval.yaml --defense sentinel

## 4 · Real Qwen3-8B
`SENTINEL_HF_QUANT=4bit` loads the same weights in NF4 so they fit a 16 GB T4 (declare it in the report). Weights load once per process.

In [ ]:
import os
os.environ['SENTINEL_HF_QUANT'] = '4bit'   # remove on L4/A100
os.makedirs('out', exist_ok=True)

In [ ]:
# (a) non-vacuity control: undefended agent must actually fall for the attack (attack_success=True)
!.venv/bin/sentinel run --scenario scenarios/public/enterprise/enterprise_direct_token_request.yaml --defense allow_all --model qwen3-8b --artifacts out/artifacts

In [ ]:
# (b) same attack, SENTINEL on
!.venv/bin/sentinel run --scenario scenarios/public/enterprise/enterprise_direct_token_request.yaml --defense sentinel --model qwen3-8b --artifacts out/artifacts

In [ ]:
# (c) benign task must still complete
!.venv/bin/sentinel run --scenario scenarios/public/enterprise/enterprise_project_status.yaml --defense sentinel --attacker none --attack-mode none --model qwen3-8b --artifacts out/artifacts

## 5 · Full public + validation evals with Qwen (slow: ~20-40 min each on a T4)

In [ ]:
!.venv/bin/sentinel eval public --defense sentinel --model qwen3-8b --artifacts out/artifacts --output out/public-sentinel-qwen.json

In [ ]:
!.venv/bin/sentinel eval validation --defense sentinel --model qwen3-8b --artifacts out/artifacts --output out/validation-sentinel-qwen.json

## 6 · Dashboard from fresh results, then download everything

In [ ]:
!.venv/bin/python scripts/build_dashboard.py --extra out --out out/sentinel-dashboard.html
import shutil
from google.colab import files
shutil.make_archive('sentinel_colab_out', 'zip', 'out')
files.download('sentinel_colab_out.zip')